# Building the Trip-Level Master Table

This notebook joins three cleaned datasets into a single trip-level master table.

| Source | Rows | Grain | Join Key |
|---|---|---|---|
| `trips_cleaned_wide.csv` | 14,859 | 1 row per trip | `trip_id` |
| `gps_health_summary.csv` | 14,859 | 1 row per trip | `trip_id` |
| `trip_positions_cleaned.csv` | ~2.35M | 1 row per GPS ping | `trip_id` (aggregated first) |

Since `trip_positions_cleaned` is at a much finer grain (many rows per trip),
I aggregated it to trip level before joining.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

# CONFIGURATION
TRIPS_PATH    = "<INPUT_DATA_DIR>/trips_cleaned_wide.csv"
GPS_PATH      = "<INPUT_DATA_DIR>/gps_health_summary.csv"
POSITIONS_PATH = "<INPUT_DATA_DIR>/trip_positions_cleaned.csv"
OUTPUT_PATH   = "<INPUT_DATA_DIR>/trip_master.csv"


## 1. Load the three datasets


In [ ]:
trips = pd.read_csv(TRIPS_PATH, low_memory=False)
gps   = pd.read_csv(GPS_PATH)
pos   = pd.read_csv(POSITIONS_PATH, low_memory=False)

print(f"trips_cleaned_wide:      {trips.shape[0]:>10,} rows  ×  {trips.shape[1]} cols")
print(f"gps_health_summary:      {gps.shape[0]:>10,} rows  ×  {gps.shape[1]} cols")
print(f"trip_positions_cleaned:  {pos.shape[0]:>10,} rows  ×  {pos.shape[1]} cols")


## 2. Verify join keys

Before joining, confirm that `trip_id` aligns across all three tables.


In [ ]:
trips_ids = set(trips["trip_id"])
gps_ids   = set(gps["trip_id"])
pos_ids   = set(pos["trip_id"])

print(f"Unique trip_ids in trips:      {len(trips_ids):,}")
print(f"Unique trip_ids in gps:        {len(gps_ids):,}")
print(f"Unique trip_ids in positions:  {len(pos_ids):,}")
print(f"\ntrips ∩ gps:        {len(trips_ids & gps_ids):,}  (should be 14,859)")
print(f"trips ∩ positions:  {len(trips_ids & pos_ids):,}  (trips with GPS data)")
print(f"trips only:         {len(trips_ids - pos_ids):,}  (trips with no GPS points  (pending trips))")


## 3. Aggregate trip positions to trip level

The raw positions table has ~2.35M rows. We summarise it to one row per `trip_id`
so it can be joined to the other trip-level tables.

| Aggregate Column | Description |
|---|---|
| `gps_point_count` | Total GPS pings for this trip |
| `first_ping_time` | Earliest timestamp |
| `last_ping_time` | Latest timestamp |
| `actual_duration_min` | Time between first and last ping (minutes) |
| `avg_gap_s` | Average seconds between consecutive pings |
| `max_gap_s` | Largest gap between consecutive pings |
| `pct_large_gap` | % of gaps exceeding 2× the median gap |
| `frozen_ping_count` | Number of pings flagged as frozen |
| `frozen_pct` | % of pings that are frozen |
| `out_of_order_count` | Pings flagged as out of order |
| `out_of_window_count` | Pings flagged as outside expected time window |


In [ ]:
# Aggregate positions to one row per trip
pos_agg = pos.groupby("trip_id").agg(
    gps_point_count      = ("position_id", "count"),
    first_ping_time      = ("timestamp", "min"),
    last_ping_time       = ("timestamp", "max"),
    avg_gap_s            = ("gap_s", "mean"),
    max_gap_s            = ("gap_s", "max"),
    frozen_ping_count    = ("is_frozen", "sum"),
    out_of_order_count   = ("is_out_of_order", "sum"),
    out_of_window_count  = ("is_out_of_window", "sum"),
).reset_index()

# Derived columns
pos_agg["actual_duration_min"] = (pos_agg["last_ping_time"] - pos_agg["first_ping_time"]) / 60
pos_agg["frozen_pct"] = (pos_agg["frozen_ping_count"] / pos_agg["gps_point_count"] * 100).round(2)

# % of large gaps (>2x median gap per trip)
median_gap = pos.groupby("trip_id")["gap_s"].median().rename("median_gap_s")
pos_with_median = pos.merge(median_gap, on="trip_id", how="left")
large_gap_counts = (
    pos_with_median[pos_with_median["gap_s"] > 2 * pos_with_median["median_gap_s"]]
    .groupby("trip_id")
    .size()
    .rename("large_gap_count")
)
pos_agg = pos_agg.merge(large_gap_counts, on="trip_id", how="left")
pos_agg["large_gap_count"] = pos_agg["large_gap_count"].fillna(0).astype(int)
pos_agg["pct_large_gap"] = (pos_agg["large_gap_count"] / pos_agg["gps_point_count"] * 100).round(2)

print(f"Aggregated positions: {pos_agg.shape[0]:,} rows  ×  {pos_agg.shape[1]} cols")
pos_agg.head(3)


## 4. Join all three tables

I used `trips_cleaned_wide` as the base and left-join the other two on `trip_id`.  
This keeps all 14,859 trips, those without GPS data get `NaN` in the position and health columns.


In [ ]:
# Join trips + gps health (1:1)
master = trips.merge(gps, on="trip_id", how="left")
print(f"After joining gps_health:    {master.shape[0]:,} rows  ×  {master.shape[1]} cols")

# Join + aggregated positions
master = master.merge(pos_agg, on="trip_id", how="left")
print(f"After joining positions_agg: {master.shape[0]:,} rows  ×  {master.shape[1]} cols")

# Sanity check: should still be 14,859 rows
assert len(master) == len(trips), f"Row count changed! Expected {len(trips):,}, got {len(master):,}"
print(f"\n✓  Row count preserved: {len(master):,} trips")

#Drop duplicate collumn (trip_positions_count); trip_positions_count = gps_point_count
master = master.drop(columns=["trip_positions_count"])


## 5. Inspect the master table


In [ ]:
master.info(show_counts=True)


In [ ]:
# Missing-value report
missing = master.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
pct = (missing / len(master) * 100).round(1)
print("Columns with missing values:\n")
print(pd.DataFrame({"missing_count": missing, "pct_missing": pct}).to_string())


In [ ]:
# Peek at a few rows
display(master.head(5))


## 6. Export


In [ ]:
master.to_csv(OUTPUT_PATH, index=False)
print(f"✓  Master table saved → {OUTPUT_PATH}")
print(f"   {master.shape[0]:,} rows  ×  {master.shape[1]} cols")
